In [1]:
!pip install ipython==8.12.0

In [2]:
!pip install datasets
!pip install einops
!pip install transformers

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from torchvision.utils import save_image, make_grid
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import random
import os
import math
from collections import namedtuple
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from functools import partial
import transformers
import matplotlib.animation
from IPython.display import HTML
from IPython.display import clear_output
import time
from google.colab import drive

import sys

In [4]:
# TODO: Mount your Google Drive; this allows the runtime environment to access your drive.
drive.mount('/content/gdrive')

# NOTE: Make sure your path does NOT include a '/' at the end!
base_dir = "/content/gdrive/MyDrive/CS4782PatchTSTReplication"
sys.path.append(base_dir)
## END TODO

Mounted at /content/gdrive


In [5]:
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

from models import PatchTST
from models.test_patch_tst import TestPatchTST
from models.tstv2 import PatchTSTTwo
from data_provider import Dataset_Custom

random.seed(0)
torch.manual_seed(0)

In [6]:

# =========================
# CONFIG (edit here)
# =========================
class Config:
    task_id = "ili_60" # Example task ID for prediction length 60
    data = "/content/gdrive/MyDrive/CS4782PatchTSTReplication/data/ETT-small/ETTh1.csv"
    features = "M"

    # ILI specific dimensions
    enc_in = 7

    # Sequence settings for ILI
    seq_len = 336
    pred_len = 96 # Common ILI prediction length is 24, 36, 48, or 60 [cite: 192]

    # Reduced model capacity to prevent overfitting
    d_model = 128
    n_heads = 16
    e_layers = 3

    # Patching stays consistent with paper defaults [cite: 197]
    patch_len = 16
    stride = 8

    # Optimization settings
    batch_size = 32 # Can be larger for ILI as it's a small dataset
    accumulation_steps = 1 # No need for accumulation with small M
    use_amp = True

    train_epochs = 10 # Small dataset may need more epochs to converge
    learning_rate = 1e-3

cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# Model
# =========================
model = PatchTSTTwo(
    M=cfg.enc_in,
    L=cfg.seq_len,
    T=cfg.pred_len,
    P=cfg.patch_len,
    S=cfg.stride,
    D=cfg.d_model,
    n_heads=cfg.n_heads,
    n_layers=cfg.e_layers
).to(device)


# =========================
# Data
# =========================
train_data = Dataset_Custom(
    data_path=cfg.data,
    flag='train',
    size=[cfg.seq_len, cfg.pred_len]
)

train_loader = DataLoader(
    train_data,
    batch_size=cfg.batch_size,
    shuffle=True
)

val_data = Dataset_Custom(
    data_path=cfg.data,
    flag='val',
    size=[cfg.seq_len, cfg.pred_len]
)

val_loader = DataLoader(
    val_data,
    batch_size=cfg.batch_size,
    shuffle=False
)


# =========================
# Optimizer / Loss
# =========================
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.train_epochs)
criterion = nn.MSELoss()


# =========================
# Training loop
# =========================
scaler = GradScaler()
model.train()

best_val_loss = float('inf')
for epoch in range(cfg.train_epochs):
    losses = []
    optimizer.zero_grad() # Reset outside the loop
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.train_epochs}")

    for i, (batch_x, batch_y) in enumerate(loop):
        batch_x = batch_x.float().to(device).permute(0, 2, 1)
        batch_y = batch_y.float().to(device)

        # 1. Mixed Precision Forward Pass
        with autocast(enabled=cfg.use_amp):
            outputs = model(batch_x)
            outputs = outputs.permute(0, 2, 1)
            # Scale loss by accumulation steps
            loss = criterion(outputs, batch_y) / cfg.accumulation_steps

        # 2. Scaled Backwards Pass
        scaler.scale(loss).backward()

        # 3. Step Optimizer every N iterations
        if (i + 1) % cfg.accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        losses.append(loss.item() * cfg.accumulation_steps)
        loop.set_postfix(loss=float(loss.item() * cfg.accumulation_steps))

    print(f"Epoch {epoch+1} | Avg Loss: {np.mean(losses):.6f}")

    # Validation Loop
    model.eval()
    val_losses = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.float().to(device).permute(0, 2, 1)
            batch_y = batch_y.float().to(device)

            outputs = model(batch_x)
            outputs = outputs.permute(0, 2, 1)

            loss = criterion(outputs, batch_y)
            val_losses.append(loss.item())

    avg_val_loss = np.mean(val_losses)
    print(f"Epoch {epoch+1} | Val Loss: {avg_val_loss:.6f}")

    scheduler.step()

    if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      torch.save(model.state_dict(), "best_patchtst.pth")
      print(f"Saved new best model | Val Loss: {avg_val_loss:.6f}")

Using device: cuda


/tmp/ipykernel_11622/1959113123.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/10:   0%|          | 0/368 [00:00<?, ?it/s]/tmp/ipykernel_11622/1959113123.py:106: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.use_amp):
Epoch 1/10: 100%|██████████| 368/368 [00:07<00:00, 50.98it/s, loss=0.306]


Epoch 1 | Avg Loss: 0.390617
Epoch 1 | Val Loss: 0.313539
Saved new best model | Val Loss: 0.313539


Epoch 2/10: 100%|██████████| 368/368 [00:05<00:00, 62.08it/s, loss=0.259]


Epoch 2 | Avg Loss: 0.320763
Epoch 2 | Val Loss: 0.348572


...

Epoch 8 | Avg Loss: 0.083217
Epoch 8 | Val Loss: 0.494674


Epoch 9/10: 100%|██████████| 368/368 [00:06<00:00, 60.99it/s, loss=0.0749]


Epoch 9 | Avg Loss: 0.076271
Epoch 9 | Val Loss: 0.489143


Epoch 10/10: 100%|██████████| 368/368 [00:10<00:00, 35.13it/s, loss=0.069]


Epoch 10 | Avg Loss: 0.072809
Epoch 10 | Val Loss: 0.493187


In [7]:
# Test Cell

model.load_state_dict(torch.load("best_patchtst.pth", map_location=device))
model.eval()

test_data = Dataset_Custom(
    data_path=cfg.data,
    flag='test',
    size=[cfg.seq_len, cfg.pred_len]
)

test_loader = DataLoader(
    test_data,
    batch_size=cfg.batch_size,
    shuffle=False
)

test_mse = []
test_mae = []

with torch.no_grad():
    for batch_x, batch_y in tqdm(test_loader, desc="Testing"):
        batch_x = batch_x.float().to(device).permute(0, 2, 1)
        batch_y = batch_y.float().to(device)

        outputs = model(batch_x)
        outputs = outputs.permute(0, 2, 1)

        mse = torch.mean((outputs - batch_y) ** 2)
        test_mse.append(mse.item())

        mae = torch.mean(torch.abs(outputs - batch_y))
        test_mae.append(mae.item())

print("\n")
print(f"Test MSE: {np.mean(test_mse):.6f}")
print(f"Test MAE: {np.mean(test_mae):.6f}")

Testing: 100%|██████████| 106/106 [00:00<00:00, 121.93it/s]



Test MSE: 0.431705
Test MAE: 0.459033


# In the Case of FOMC rate

In [8]:
# =========================
# CONFIG (edit here)
# =========================
class Config:
    task_id = "fomc_96"
    data = "/content/gdrive/MyDrive/CS4782PatchTSTReplication/data/fomc/fomc_macro.csv"
    features = "M"

    # 7 macro series: fed_funds, cpi, unemployment, core_pce,
    #                 yield_spread, housing_starts, industrial_production
    enc_in = 7

    # Monthly data — use ILI-style short lookback (104 months ≈ 8.5 years)
    # pred_len 24 = forecasting 2 years ahead
    seq_len = 104
    pred_len = 24

    # Slightly reduced capacity for small monthly dataset
    d_model = 64
    n_heads = 8
    e_layers = 3

    patch_len = 16
    stride = 8

    batch_size = 16          # dataset is small (~400 rows after dropna)
    accumulation_steps = 1
    use_amp = True

    train_epochs = 30        # more epochs needed for small dataset
    learning_rate = 1e-3

cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# Model
# =========================
model = PatchTSTTwo(
    M=cfg.enc_in,
    L=cfg.seq_len,
    T=cfg.pred_len,
    P=cfg.patch_len,
    S=cfg.stride,
    D=cfg.d_model,
    n_heads=cfg.n_heads,
    n_layers=cfg.e_layers
).to(device)


# =========================
# Data
# =========================
train_data = Dataset_Custom(
    data_path=cfg.data,
    flag='train',
    size=[cfg.seq_len, cfg.pred_len]
)

train_loader = DataLoader(
    train_data,
    batch_size=cfg.batch_size,
    shuffle=True
)

val_data = Dataset_Custom(
    data_path=cfg.data,
    flag='val',
    size=[cfg.seq_len, cfg.pred_len]
)

val_loader = DataLoader(
    val_data,
    batch_size=cfg.batch_size,
    shuffle=False
)


# =========================
# Optimizer / Loss
# =========================
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.train_epochs)
criterion = nn.MSELoss()


# =========================
# Training loop
# =========================
scaler = GradScaler()
model.train()

best_val_loss = float('inf')
for epoch in range(cfg.train_epochs):
    model.train()
    losses = []
    optimizer.zero_grad()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.train_epochs}")

    for i, (batch_x, batch_y) in enumerate(loop):
        batch_x = batch_x.float().to(device).permute(0, 2, 1)
        batch_y = batch_y.float().to(device)

        with autocast(enabled=cfg.use_amp):
            outputs = model(batch_x)
            outputs = outputs.permute(0, 2, 1)
            loss = criterion(outputs, batch_y) / cfg.accumulation_steps

        scaler.scale(loss).backward()

        if (i + 1) % cfg.accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        losses.append(loss.item() * cfg.accumulation_steps)
        loop.set_postfix(loss=float(loss.item() * cfg.accumulation_steps))

    print(f"Epoch {epoch+1} | Train Loss: {np.mean(losses):.6f}")

    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.float().to(device).permute(0, 2, 1)
            batch_y = batch_y.float().to(device)
            outputs = model(batch_x)
            outputs = outputs.permute(0, 2, 1)
            val_losses.append(criterion(outputs, batch_y).item())

    avg_val_loss = np.mean(val_losses)
    print(f"Epoch {epoch+1} | Val Loss:   {avg_val_loss:.6f}")

    scheduler.step()

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_patchtst_fomc.pth")
        print(f"  --> Saved new best model (Val Loss: {avg_val_loss:.6f})")

Using device: cuda


/tmp/ipykernel_11622/1087943746.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/30:   0%|          | 0/11 [00:00<?, ?it/s]/tmp/ipykernel_11622/1087943746.py:106: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.use_amp):
Epoch 1/30: 100%|██████████| 11/11 [00:00<00:00, 67.91it/s, loss=0.495]


Epoch 1 | Train Loss: 0.700045
Epoch 1 | Val Loss:   0.679601
  --> Saved new best model (Val Loss: 0.679601)


Epoch 2/30: 100%|██████████| 11/11 [00:00<00:00, 70.03it/s, loss=0.304]


Epoch 2 | Train Loss: 0.387620
Epoch 2 | Val Loss:   0.231427
  --> Saved new best model (Val Loss: 0.231427)


...

Epoch 28 | Train Loss: 0.116333
Epoch 28 | Val Loss:   0.627526


Epoch 29/30: 100%|██████████| 11/11 [00:00<00:00, 71.86it/s, loss=0.139]


Epoch 29 | Train Loss: 0.116238
Epoch 29 | Val Loss:   0.629846


Epoch 30/30: 100%|██████████| 11/11 [00:00<00:00, 78.10it/s, loss=0.125]

Epoch 30 | Train Loss: 0.119857
Epoch 30 | Val Loss:   0.629551


In [9]:
# =========================
# Test Cell
# =========================
model.load_state_dict(torch.load("best_patchtst_fomc.pth", map_location=device))
model.eval()

test_data = Dataset_Custom(
    data_path=cfg.data,
    flag='test',
    size=[cfg.seq_len, cfg.pred_len]
)

test_loader = DataLoader(
    test_data,
    batch_size=cfg.batch_size,
    shuffle=False
)

test_mse = []
test_mae = []

with torch.no_grad():
    for batch_x, batch_y in tqdm(test_loader, desc="Testing"):
        batch_x = batch_x.float().to(device).permute(0, 2, 1)
        batch_y = batch_y.float().to(device)

        outputs = model(batch_x)
        outputs = outputs.permute(0, 2, 1)

        test_mse.append(torch.mean((outputs - batch_y) ** 2).item())
        test_mae.append(torch.mean(torch.abs(outputs - batch_y)).item())

print(f"\nTest MSE: {np.mean(test_mse):.6f}")
print(f"Test MAE: {np.mean(test_mae):.6f}")

Testing: 100%|██████████| 4/4 [00:00<00:00, 347.52it/s]


Test MSE: 0.965230
Test MAE: 0.613825


In [10]:
results = {}

for pred_len in [12, 24, 36]:
    cfg.pred_len = pred_len

    # Reinitialize model for each horizon
    model = PatchTSTTwo(
        M=cfg.enc_in,
        L=cfg.seq_len,
        T=cfg.pred_len,
        P=cfg.patch_len,
        S=cfg.stride,
        D=cfg.d_model,
        n_heads=cfg.n_heads,
        n_layers=cfg.e_layers
    ).to(device)

    # Data
    train_data = Dataset_Custom(data_path=cfg.data, flag='train', size=[cfg.seq_len, cfg.pred_len])
    val_data   = Dataset_Custom(data_path=cfg.data, flag='val',   size=[cfg.seq_len, cfg.pred_len])
    test_data  = Dataset_Custom(data_path=cfg.data, flag='test',  size=[cfg.seq_len, cfg.pred_len])

    train_loader = DataLoader(train_data, batch_size=cfg.batch_size, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=cfg.batch_size, shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=cfg.batch_size, shuffle=False)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.train_epochs)
    criterion = nn.MSELoss()
    scaler    = torch.amp.GradScaler(enabled=cfg.use_amp) # Updated to use torch.amp.GradScaler

    best_val_loss = float('inf')
    ckpt_path = f"best_patchtst_ili_{pred_len}.pth"

    # Training
    for epoch in range(cfg.train_epochs):
        model.train()
        losses = []
        optimizer.zero_grad()

        for i, (batch_x, batch_y) in enumerate(train_loader):
            batch_x = batch_x.float().to(device).permute(0, 2, 1)
            batch_y = batch_y.float().to(device)

            with torch.amp.autocast(device_type=str(device), enabled=cfg.use_amp): # Updated to use torch.amp.autocast
                outputs = model(batch_x).permute(0, 2, 1)
                loss = criterion(outputs, batch_y) / cfg.accumulation_steps

            scaler.scale(loss).backward()

            if (i + 1) % cfg.accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            losses.append(loss.item() * cfg.accumulation_steps)

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.float().to(device).permute(0, 2, 1)
                batch_y = batch_y.float().to(device)
                outputs = model(batch_x).permute(0, 2, 1)
                val_losses.append(criterion(outputs, batch_y).item())

        avg_val_loss = np.mean(val_losses)
        scheduler.step()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), ckpt_path)

        print(f"pred_len={pred_len} | Epoch {epoch+1}/{cfg.train_epochs} | "
              f"Train: {np.mean(losses):.4f} | Val: {avg_val_loss:.4f}")

    # Testing
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    test_mse, test_mae = [], []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.float().to(device).permute(0, 2, 1)
            batch_y = batch_y.float().to(device)
            outputs = model(batch_x).permute(0, 2, 1)
            test_mse.append(torch.mean((outputs - batch_y) ** 2).item())
            test_mae.append(torch.mean(torch.abs(outputs - batch_y)).item())

    results[pred_len] = {
        'MSE': round(np.mean(test_mse), 3),
        'MAE': round(np.mean(test_mae), 3)
    }
    print(f"\n>>> pred_len={pred_len} | Test MSE: {results[pred_len]['MSE']:.3f} | "
          f"Test MAE: {results[pred_len]['MAE']:.3f}\n")

# Print final table
print(f"\n{'Pred Len':>10} | {'MSE':>8} | {'MAE':>8}")
print("-" * 35)
for pl, metrics in results.items():
    print(f"{pl:>10} | {metrics['MSE']:>8.3f} | {metrics['MAE']:>8.3f}")

pred_len=12 | Epoch 1/30 | Train: 0.4625 | Val: 0.2862
pred_len=12 | Epoch 2/30 | Train: 0.2115 | Val: 0.1982
pred_len=12 | Epoch 3/30 | Train: 0.1642 | Val: 0.0772
pred_len=12 | Epoch 4/30 | Train: 0.1506 | Val: 0.1101
pred_len=12 | Epoch 5/30 | Train: 0.1294 | Val: 0.1163
pred_len=12 | Epoch 6/30 | Train: 0.1307 | Val: 0.1087
pred_len=12 | Epoch 7/30 | Train: 0.1162 | Val: 0.0969
pred_len=12 | Epoch 8/30 | Train: 0.1073 | Val: 0.1151
pred_len=12 | Epoch 9/30 | Train: 0.1007 | Val: 0.1566
pred_len=12 | Epoch 10/30 | Train: 0.1026 | Val: 0.1223
pred_len=12 | Epoch 11/30 | Train: 0.0954 | Val: 0.1561
pred_len=12 | Epoch 12/30 | Train: 0.0934 | Val: 0.0992
pred_len=12 | Epoch 13/30 | Train: 0.0943 | Val: 0.0935
pred_len=12 | Epoch 14/30 | Train: 0.0912 | Val: 0.1219
pred_len=12 | Epoch 15/30 | Train: 0.0820 | Val: 0.1441
pred_len=12 | Epoch 16/30 | Train: 0.0841 | Val: 0.1316
pred_len=12 | Epoch 17/30 | Train: 0.0799 | Val: 0.1496
pred_len=12 | Epoch 18/30 | Train: 0.0793 | Val: 0.1361
p